# How to Detect Data Drift

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/detect_data_drift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fdetect_data_drift.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/detect_data_drift.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


`DriftDetector` monitors whether incoming data has shifted from the distribution your pipeline was trained on. Critical for production quantum ML systems where circuit parameters are fixed after training.

In [ ]:
!pip install -q quprep

In [ ]:
import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning

print(f"quprep {qd.__version__}")

rng = np.random.default_rng(42)
X_train = rng.normal(loc=0.5, scale=0.1, size=(100, 4))
ds_train = qd.NumpyIngester().load(X_train)

detector = qd.DriftDetector()
detector.fit(ds_train)
print(f"Reference mean : {X_train.mean(axis=0).round(3)}")

## 1. No drift (same distribution)

In [ ]:
X_same = rng.normal(loc=0.5, scale=0.1, size=(50, 4))
ds_same = qd.NumpyIngester().load(X_same)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    report_no_drift = detector.check(ds_same)

print(f"Drift detected   : {report_no_drift.overall_drift}")
print(f"Features drifted : {report_no_drift.n_features_drifted}")

## 2. Clear drift (shifted distribution)

In [ ]:
X_drift = rng.normal(loc=2.0, scale=0.3, size=(50, 4))  # mean shifted 0.5 → 2.0
ds_drift = qd.NumpyIngester().load(X_drift)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    report_drift = detector.check(ds_drift)

print(f"Drift detected   : {report_drift.overall_drift}")
print(f"Features drifted : {report_drift.n_features_drifted}")
if report_drift.drifted_features:
    print(f"Drifted features : {report_drift.drifted_features}")

## 3. Drift detector inside a Pipeline

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    pipeline = qd.Pipeline(
        normalizer=qd.Scaler(strategy="minmax_pi"),
        encoder=qd.AngleEncoder(),
        drift_detector=qd.DriftDetector(),
    )
    result_train = pipeline.fit_transform(ds_train)
    result_new   = pipeline.transform(ds_drift)

train_drift = result_train.drift_report.overall_drift if result_train.drift_report else "n/a (fit)"
print(f"   Training drift   : {train_drift}")
print(f"New data drift   : {result_new.drift_report.overall_drift}")
print(f"Features drifted : {result_new.drift_report.n_features_drifted}")